# Exercise: the distillation loss for one example

**Goal:** compute the top-K self-distillation loss for one example from `runs/distill/examples.jsonl`, by hand, step by step.

- **Teacher:** Qwen3.5-9B with the hint in its context (`teacher_prompt`).
- **Student:** the same model without the hint (`student_prompt`).
- Both read the same **target**: the tokens the model actually wrote at the hint turn.

At every target position $i$, the teacher's top-K tokens give a distribution $p_T(\cdot \mid i)$. The loss pushes the student's distribution toward it:

$$\mathcal{L} = \frac{1}{T}\sum_{i=1}^{T} \sum_{k \in \text{top-}K_T(i)} p_T(k \mid i)\,\big(\log p_T(k \mid i) - \log p_S(k \mid i)\big)$$

That's KL(teacher ‖ student) restricted to the teacher's top-K tokens, averaged over the $T$ target positions.

**Steps**
1. Build the teacher and student sequences, and find where the target sits in each.
2. Get the teacher's top-K tokens and log-probs at the target positions (Tinker).
3. Get the student's log-probs for those same tokens: a Tinker training-client forward pass on a training datum.
4. Compute the loss.
5. Look at where the teacher and student disagree most.
6. *(Optional)* Compare with Tinker's own `cross_entropy` loss on the same datum.

Each step has a **TODO** cell for you and a **check** cell after it. Run the check: it tells you whether your code is right.

**Cost:** the default example has an 11.7k-token context and 107 target tokens, so each Tinker call is about a cent.

In [ ]:
# Setup (given). Run this first.
import json
import os
from pathlib import Path
import sys

import numpy as np
import tinker
import torch

# Jupyter runs from notebooks/; make the repo root the working directory and importable.
REPO = Path.cwd() if (Path.cwd() / "sd").exists() else Path.cwd().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from sd.renderer import get_agent_renderer


# TINKER_API_KEY from the environment, or from the gitignored .env
if "TINKER_API_KEY" not in os.environ and (REPO / ".env").exists():
    for line in (REPO / ".env").read_text().splitlines():
        if line.strip() and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())
assert "TINKER_API_KEY" in os.environ, "set TINKER_API_KEY or create .env"

MODEL = "Qwen/Qwen3.5-9B"
K = 20

examples = [json.loads(l) for l in open("runs/distill/examples.jsonl")]
example = next(e for e in examples if e["instance_id"] == "sphinx-doc__sphinx-8638")  # smallest context: cheap

renderer = get_agent_renderer(MODEL)
tok = renderer.tokenizer
service = tinker.ServiceClient()
sampling_client = service.create_sampling_client(base_model=MODEL)       # teacher (step 2)
training_client = service.create_lora_training_client(base_model=MODEL)  # student forward pass (step 3)

print(example["instance_id"], "| hint turn", example["hint_turn"])
print("student prompt", len(example["student_prompt"]), "tokens | teacher prompt", len(example["teacher_prompt"]),
      "| target", len(example["target"]))
print("hint:", example["hint"])
print("target starts:", repr(tok.decode(example["target"][:40])))

## Step 1: build the sequences

The model reads the prompt followed by the target, so the full sequences are:

- `student_seq = student_prompt + target`
- `teacher_seq = teacher_prompt + target`

The target is the **last `T` tokens of both**, but at different absolute positions, because the teacher's prompt is longer by the hint message (about 42 tokens).

Compute:
- `T`: the number of target tokens
- `student_pos`: the list of positions of the target tokens inside `student_seq`
- `teacher_pos`: the same inside `teacher_seq`

For example, if `student_seq` has length 10 and `T = 3`, then `student_pos == [7, 8, 9]`.

In [ ]:
# TODO
target = example["target"]
T = ...
student_seq = ...
teacher_seq = ...
student_pos = ...
teacher_pos = ...

In [ ]:
# Check step 1
assert T == len(example["target"])
assert len(student_seq) == len(example["student_prompt"]) + T and len(teacher_seq) == len(example["teacher_prompt"]) + T
assert [student_seq[p] for p in student_pos] == target and [teacher_seq[p] for p in teacher_pos] == target
assert student_pos[-1] == len(student_seq) - 1 and teacher_pos[-1] == len(teacher_seq) - 1
print("step 1 OK")

## Step 2: the teacher's top-K at the target positions

Ask Tinker for top-K **prompt** log-probs over the whole `teacher_seq`. We don't need any generation, so `max_tokens=1`.

```python
response = sampling_client.sample(
    prompt=tinker.ModelInput.from_ints(teacher_seq),
    num_samples=1,
    sampling_params=tinker.SamplingParams(max_tokens=1),
    include_prompt_logprobs=True,   # required: without it, no top-K comes back
    topk_prompt_logprobs=K,
).result()
response.topk_prompt_logprobs_np.token_ids  # int32,   shape (len(teacher_seq), K)
response.topk_prompt_logprobs_np.logprobs   # float32, shape (len(teacher_seq), K)
```

**Indexing:** row `i` is the teacher's distribution for the token **at position `i`**, given the tokens before it. Row 0 is empty, since nothing comes before it.

Compute, as torch tensors of shape `(T, K)`:
- `teacher_ids`: the teacher's top-K token IDs at each target position
- `teacher_lp`: their log-probs

In [ ]:
# TODO
response_t = ...
teacher_ids = ...
teacher_lp = ...

In [ ]:
# Check step 2
assert teacher_ids.shape == (T, K) and teacher_lp.shape == (T, K)
assert torch.all(teacher_lp <= 0) and torch.all(teacher_lp[:, :-1] >= teacher_lp[:, 1:]), "rows should be sorted, best first"
in_topk = (teacher_ids == torch.tensor(target).unsqueeze(1)).any(1).float().mean().item()
mass = teacher_lp.exp().sum(1)
print(f"step 2 OK | the written token is in the teacher's top-{K} at {in_topk:.0%} of positions | "
      f"top-{K} covers {mass.mean():.1%} of the teacher's probability on average (min {mass.min():.1%})")


## Step 3: the student's log-probs for the same tokens

We need the **student's** log-prob for each of the teacher's K tokens at each target position. The sampler can't do this here: it scores at most 20 distinct token IDs per request, and the teacher's top-20 over a whole target involve about 1,000.

The **training client's forward pass** can, and it's the same data format training uses. Build a `tinker.Datum`:

```python
datum = tinker.Datum(
    model_input=tinker.ModelInput.from_ints(student_seq[:-1]),       # the model predicts token i+1 from tokens <= i
    loss_fn_inputs={
        "target_tokens": tinker.TensorData.from_torch(target_tokens),  # int64, shape (len(student_seq) - 1, K)
        "weights": tinker.TensorData.from_torch(weights),              # float32, same shape
    },
)
out = training_client.forward([datum], loss_fn="cross_entropy").result()   # forward only: no weight update
out.loss_fn_outputs[0]["logprobs"].to_torch()   # shape (len(student_seq) - 1, K): the student's log-prob of each target cell
```

**Indexing:** **row `i` predicts the token at position `i + 1`.** So the target token at position `p` lives in row `p - 1`. (Step 2 was different: there, row `i` was position `i`.)

Fill `target_tokens` with the teacher's top-K IDs on the target rows. Other rows can hold anything, e.g. zeros. Set `weights` to the teacher's top-K **probabilities**, renormalized to sum to 1 per row, on the target rows, and `0` elsewhere. The weights don't affect the log-probs you read here, but step 6 uses this same datum for training's loss.

Compute `student_lp`, shape `(T, K)`: the student's log-probs for `teacher_ids`.


In [ ]:
# TODO
target_tokens = ...
weights = ...
datum = ...
out = ...
student_lp = ...

In [ ]:
# Check step 3
assert student_lp.shape == (T, K) and torch.all(student_lp <= 0)
# The student's log-prob of the token it actually wrote should match what vLLM recorded during the rollout.
match = teacher_ids == torch.tensor(target).unsqueeze(1)              # where the written token sits in each row
has = match.any(1)
written_lp = student_lp[has, match[has].float().argmax(1)]
diff = (written_lp - torch.tensor(example["target_logprobs"])[has]).abs()
print(f"step 3 OK | student vs recorded log-prob of the written token ({has.sum()}/{T} positions): "
      f"median |diff| {diff.median():.4f}, max {diff.max():.3f}")
assert diff.median() < 0.05, "student log-probs don't match the rollout: check the row/position offset"

## Step 4: the loss

Implement `topk_kl(teacher_lp, student_lp)`. Both inputs have shape `(T, K)` and hold log-probs over the **same** K tokens per row. It returns a tensor of shape `(T,)`: the KL at each position.

1. Renormalize the teacher over its K tokens: `p_T = softmax(teacher_lp)` per row, so each row sums to 1.
2. Renormalize the student over the **same** K tokens: `log p_S = log_softmax(student_lp)` per row. This makes each row a proper distribution, so the KL is ≥ 0.
3. Per row: $\sum_k p_T(k)\,(\log p_T(k) - \log p_S(k))$.

Then `loss = per_position.mean()`, the average over target positions.

*(The cookbook's version skips step 2 and trains the student's full-vocabulary log-probs toward the teacher's top-K probabilities with cross-entropy. That version is what Step 6 sets up.)*

In [ ]:
# TODO
def topk_kl(teacher_lp: torch.Tensor, student_lp: torch.Tensor) -> torch.Tensor:
    ...

per_position = topk_kl(teacher_lp, student_lp)
loss = per_position.mean()

In [ ]:
# Check step 4
t = torch.log(torch.tensor([[0.5, 0.3, 0.2]])); s = torch.log(torch.tensor([[0.2, 0.3, 0.5]]))
assert torch.allclose(topk_kl(t, s), torch.tensor([0.274887]), atol=1e-5), topk_kl(t, s)   # worked out by hand
assert torch.allclose(topk_kl(t, t), torch.zeros(1), atol=1e-6)                             # identical -> 0
assert torch.allclose(topk_kl(t, s + 3.0), topk_kl(t, s), atol=1e-5)                        # renormalized: a constant shift doesn't matter
assert per_position.shape == (T,) and torch.all(per_position >= -1e-6)
print(f"step 4 OK | loss {loss.item():.4f} | per position: median {per_position.median():.4f}, max {per_position.max():.3f}")

## Step 5: where do the teacher and student disagree?

Using `per_position`, find the **10 positions with the highest KL**. For each, print:
- the position within the target, and the written token (decode it with `tok.decode([id])`)
- the KL
- the teacher's top 3 tokens with their probabilities, next to the student's probabilities for those same tokens

Questions to answer:
- Are the big disagreements at the start of the turn, for example its first thinking tokens, or spread out?
- Does the teacher's preferred alternative look like wrapping up (*the fix works*, *review the diff*, *submit*) while the written token keeps exploring (*also*, *test*, *check*)?
- *Bonus:* plot `per_position` against position within the target.

In [ ]:
# TODO

## Step 6 (optional): compare with Tinker's own loss

Training will call `training_client.forward_backward([datum], loss_fn="cross_entropy")` on datums like the one from step 3. That loss is cross-entropy with soft targets: per cell, `-weight * logprob`, summed. The forward pass in step 3 already computed it.

- Read `out.loss_fn_outputs[0]["elementwise_loss"]` and `out.metrics["loss:sum"]`.
- Recompute the same number yourself from `student_lp` and the teacher's renormalized probabilities.
- How does it relate to your KL from step 4? Cross-entropy = KL + the teacher's entropy, which is a constant for a fixed teacher. One more difference: Tinker's version uses the student's **full-vocabulary** log-probs, without renormalizing over the K tokens.

For reference, the cookbook builds the same datum in `_collect_topk_for_datum` in `.venv/lib/python3.12/site-packages/tinker_cookbook/distillation/train_off_policy.py`.

In [ ]:
# TODO (optional)